In [0]:
from pyspark.sql import functions as F

SILVER_TABLE = "silver_flights"

print(f"Reading from: {SILVER_TABLE}")

Reading from: silver_flights


In [0]:
silver = spark.table(SILVER_TABLE)
print(f"Silver rows: {silver.count():,}")
silver.printSchema()

Silver rows: 302,192
root
 |-- Reporting_Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- Route: string (nullable = true)
 |-- ArrDel15: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- IsWeekend: integer (nullable = true)
 |-- IsFixedHoliday: integer (nullable = true)
 |-- IsHolidayWindow: integer (nullable = true)
 |-- DepHour: integer (nullable = true)
 |-- DepTimeCategory: string (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- flight_hour_ts: timestamp (nullable = true)
 |-- _silver_ingested_at: timestamp (nullable = true)



In [0]:
gold_features = silver.select(
    F.col("ArrDel15").alias("label"),
    "Reporting_Airline",
    "Origin",
    "Dest",
    "DepTimeCategory",
    "Month",
    "DayOfWeek",
    "DepHour",
    "IsWeekend",
    "IsFixedHoliday",
    "IsHolidayWindow",
    "flight_date",
    "flight_hour_ts"
)

print(f"Feature table rows: {gold_features.count():,}")
gold_features.printSchema()

(
    gold_features.write
                 .format("delta")
                 .mode("overwrite")
                 .option("overwriteSchema", "true")
                 .saveAsTable("gold_flight_features")
)

print("Wrote Delta table: gold_flight_features")

Feature table rows: 302,192
root
 |-- label: integer (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DepTimeCategory: string (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- DepHour: integer (nullable = true)
 |-- IsWeekend: integer (nullable = true)
 |-- IsFixedHoliday: integer (nullable = true)
 |-- IsHolidayWindow: integer (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- flight_hour_ts: timestamp (nullable = true)

Wrote Delta table: gold_flight_features


In [0]:
carrier_perf = spark.sql(f"""
    SELECT Reporting_Airline AS carrier,
           COUNT(*)                                                AS total_flights,
           SUM(ArrDel15)                                           AS delayed_flights,
           ROUND(100.0 * SUM(ArrDel15) / COUNT(*), 2)              AS delay_rate_pct
    FROM {SILVER_TABLE}
    GROUP BY Reporting_Airline
    ORDER BY delay_rate_pct DESC
""")

(
    carrier_perf.write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable("gold_carrier_performance")
)

carrier_perf.show(20, truncate=False)

+-------+-------------+---------------+--------------+
|carrier|total_flights|delayed_flights|delay_rate_pct|
+-------+-------------+---------------+--------------+
|HA     |364          |114            |31.32         |
|OH     |6876         |2153           |31.31         |
|F9     |8145         |2488           |30.55         |
|G4     |3548         |1045           |29.45         |
|AS     |3694         |1052           |28.48         |
|B6     |52747        |14165          |26.85         |
|UA     |12379        |3061           |24.73         |
|AA     |34995        |8355           |23.87         |
|DL     |64099        |14877          |23.21         |
|YX     |67121        |15474          |23.05         |
|MQ     |4701         |1070           |22.76         |
|OO     |7280         |1643           |22.57         |
|WN     |25913        |5662           |21.85         |
|NK     |6561         |1338           |20.39         |
|9E     |3769         |756            |20.06         |
+-------+-

In [0]:
origin_perf = spark.sql(f"""
    SELECT Origin                                                  AS origin_airport,
           COUNT(*)                                                AS total_flights,
           SUM(ArrDel15)                                           AS delayed_flights,
           ROUND(100.0 * SUM(ArrDel15) / COUNT(*), 2)              AS delay_rate_pct
    FROM {SILVER_TABLE}
    GROUP BY Origin
    ORDER BY total_flights DESC
""")

(
    origin_perf.write
               .format("delta")
               .mode("overwrite")
               .option("overwriteSchema", "true")
               .saveAsTable("gold_origin_performance")
)

origin_perf.show(20, truncate=False)

+--------------+-------------+---------------+--------------+
|origin_airport|total_flights|delayed_flights|delay_rate_pct|
+--------------+-------------+---------------+--------------+
|LGA           |127721       |32587          |25.51         |
|JFK           |102767       |24808          |24.14         |
|BUF           |20193        |4254           |21.07         |
|ALB           |12241        |2659           |21.72         |
|HPN           |10587        |2371           |22.40         |
|SYR           |10333        |2390           |23.13         |
|ROC           |10046        |2256           |22.46         |
|ISP           |5186         |1207           |23.27         |
|ELM           |1215         |224            |18.44         |
|SWF           |603          |169            |28.03         |
|PBG           |539          |147            |27.27         |
|IAG           |464          |137            |29.53         |
|BGM           |234          |36             |15.38         |
|ITH    

In [0]:
hourly_perf = spark.sql(f"""
    SELECT DepHour                                                 AS dep_hour,
           COUNT(*)                                                AS total_flights,
           SUM(ArrDel15)                                           AS delayed_flights,
           ROUND(100.0 * SUM(ArrDel15) / COUNT(*), 2)              AS delay_rate_pct
    FROM {SILVER_TABLE}
    GROUP BY DepHour
    ORDER BY dep_hour
""")

(
    hourly_perf.write
               .format("delta")
               .mode("overwrite")
               .option("overwriteSchema", "true")
               .saveAsTable("gold_hourly_delay_pattern")
)

hourly_perf.show(30, truncate=False)

+--------+-------------+---------------+--------------+
|dep_hour|total_flights|delayed_flights|delay_rate_pct|
+--------+-------------+---------------+--------------+
|5       |5221         |595            |11.40         |
|6       |28525        |3442           |12.07         |
|7       |24142        |3130           |12.96         |
|8       |19730        |2979           |15.10         |
|9       |17550        |2952           |16.82         |
|10      |14031        |2567           |18.30         |
|11      |21694        |4433           |20.43         |
|12      |19341        |4005           |20.71         |
|13      |16401        |3911           |23.85         |
|14      |16713        |4898           |29.31         |
|15      |19968        |6520           |32.65         |
|16      |16899        |5446           |32.23         |
|17      |21741        |7530           |34.64         |
|18      |18158        |6496           |35.77         |
|19      |16136        |5408           |33.52   

In [0]:
monthly_trend = spark.sql(f"""
    SELECT YEAR(flight_date)                                       AS year,
           MONTH(flight_date)                                      AS month,
           COUNT(*)                                                AS total_flights,
           SUM(ArrDel15)                                           AS delayed_flights,
           ROUND(100.0 * SUM(ArrDel15) / COUNT(*), 2)              AS delay_rate_pct
    FROM {SILVER_TABLE}
    GROUP BY YEAR(flight_date), MONTH(flight_date)
    ORDER BY year, month
""")

(
    monthly_trend.write
                 .format("delta")
                 .mode("overwrite")
                 .option("overwriteSchema", "true")
                 .saveAsTable("gold_monthly_trend")
)

monthly_trend.show(15, truncate=False)

+----+-----+-------------+---------------+--------------+
|year|month|total_flights|delayed_flights|delay_rate_pct|
+----+-----+-------------+---------------+--------------+
|2024|12   |29067        |7036           |24.21         |
|2025|1    |23664        |5156           |21.79         |
|2025|2    |22243        |5963           |26.81         |
|2025|3    |25180        |4865           |19.32         |
|2025|4    |25365        |4796           |18.91         |
|2025|5    |25645        |6553           |25.55         |
|2025|6    |24692        |7671           |31.07         |
|2025|7    |25020        |7765           |31.04         |
|2025|8    |25864        |5264           |20.35         |
|2025|9    |24712        |4251           |17.20         |
|2025|10   |26044        |7142           |27.42         |
|2025|11   |24696        |6791           |27.50         |
+----+-----+-------------+---------------+--------------+



In [0]:
spark.sql("SHOW TABLES").filter(F.col("tableName").startswith("gold")).show(truncate=False)

+--------+-------------------------+-----------+
|database|tableName                |isTemporary|
+--------+-------------------------+-----------+
|default |gold_carrier_performance |false      |
|default |gold_flight_features     |false      |
|default |gold_hourly_delay_pattern|false      |
|default |gold_monthly_trend       |false      |
|default |gold_origin_performance  |false      |
+--------+-------------------------+-----------+

